# DS2002 · Weather API Join

**Studio — 2026-10-21 · Fall 2026**  
**Class time:** 45 minutes

---

## The join that eats a week of your project

Question 3 needs weather next to sales, one row per day. It sounds like a five-minute merge. In practice it is where teams lose days, and always to the same four problems:

1. Sales dates are timestamps; weather dates are dates. They never match.
2. One side is a string, the other a datetime. They never match, and pandas does not warn you.
3. The API returns a full calendar range; your sales only cover some of those days. An inner join silently discards the rest.
4. The correlation comes out strong on three data points, and somebody puts it in a deck.

Today you hit all four deliberately on fourteen rows, so you recognize them at a hundred thousand.

In [ ]:
import pandas as pd, numpy as np, requests, time

# Two weeks of daily orders around a landfall. Note: these are timestamps, not dates.
sales = pd.DataFrame({
    'ts': pd.to_datetime([
        '2024-09-01 08:14', '2024-09-02 09:02', '2024-09-03 10:41',
        '2024-09-04 08:33', '2024-09-05 11:20', '2024-09-06 07:55',
        '2024-09-07 09:10', '2024-09-08 12:02', '2024-09-09 08:44',
        '2024-09-10 10:15', '2024-09-11 09:30', '2024-09-12 08:05',
        '2024-09-13 11:45', '2024-09-14 09:22']),
    'orders': [100, 110, 95, 105, 120, 340, 520, 480, 90, 100, 110, 105, 98, 102],
})
LANDFALL = pd.Timestamp('2024-09-07')
sales.head()

### Problem 1 — a timestamp is not a date

Watch this fail, because it will fail exactly this way in your project.

In [ ]:
weather_dates = pd.to_datetime(['2024-09-06', '2024-09-07', '2024-09-08'])
tiny = pd.DataFrame({'date': weather_dates, 'rain_mm': [5.0, 60.0, 20.0]})

broken = sales.merge(tiny, left_on='ts', right_on='date', how='inner')
print('rows matched:', len(broken))

Zero. `2024-09-07 09:10` is not equal to `2024-09-07 00:00`, and no amount of staring at the two tables will show you that, because they print identically at a glance.

**TODO:** add a `date` column to `sales` that is the timestamp truncated to midnight, then redo that merge and confirm you get 3 rows.

In [ ]:
# TODO: sales['date'] = ...   (hint: .dt.normalize())
# TODO: merge on 'date' and print the row count

### Problem 2 — a string is not a datetime

The API hands back `time` as text. Merging text against datetimes gives you zero matches or, worse, a `ValueError` you will spend twenty minutes reading.

In [ ]:
api_shape = pd.DataFrame({
    'time': ['2024-09-06', '2024-09-07', '2024-09-08'],   # strings!
    'rain_mm': [5.0, 60.0, 20.0],
})
print('api time dtype:  ', api_shape['time'].dtype)
print('sales ts dtype:  ', sales['ts'].dtype)
print()
print('Rule: convert on the API side immediately, before anything else touches it.')

### Build 1 — the fetch

**TODO:** write `daily_weather(start, end, lat, lon)` for Orlando (28.54, -81.38). Requirements, all of which you have built before:

- daily `precipitation_sum` and `windspeed_10m_max`
- a timeout, and a retry on 429 or 5xx
- convert `time` to a datetime column named `date` **inside the function**
- a `source` column so a fallback is never mistaken for real data

In [ ]:
BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def daily_weather(start, end, lat=28.54, lon=-81.38, tries=3):
    # TODO: build params, request with retries, raise on non-retryable status
    # TODO: frame = pd.DataFrame(r.json()['daily'])
    # TODO: rename 'time' -> 'date', convert to datetime, add source='open-meteo'
    pass

FALLBACK = pd.DataFrame({
    'date': pd.date_range('2024-09-01', periods=14),
    'precipitation_sum': [0, 0, 2, 1, 8, 45, 120, 30, 2, 0, 0, 1, 0, 0],
    'windspeed_10m_max': [12, 14, 15, 18, 26, 44, 71, 38, 20, 15, 13, 14, 12, 11],
    'source': 'fallback sample',
})

# TODO: try the real call, fall back to FALLBACK, and print which one you got

### Build 2 — the merge, validated

**TODO:** join weather onto sales. Use a left join so no sales day disappears, and `indicator=True` so you can see what matched.

Then assert three things: the row count still equals the number of sales days, nothing is `right_only`, and the orders total is unchanged.

In [ ]:
# TODO

### Problem 3 — the days that did not match

**TODO:** if any sales day came back with no weather, you need to know which and decide what to do. Print the unmatched dates. Then answer: is dropping them acceptable for Question 3, and why?

In [ ]:
# TODO

**My decision:** _..._

### Build 3 — the baseline, which is what makes a surge a surge

"Sales were 520 on the 7th" means nothing alone. "Sales were 520 against a 14-day baseline of 104" is a finding.

**TODO:** define a baseline window (days more than 3 days before landfall), a surge window (the 3 days up to and including landfall), and print the mean of each plus the ratio.

In [ ]:
# TODO: baseline = merged[...]
# TODO: surge = merged[...]
# TODO: print both means and the multiple

### Problem 4 — the correlation you should not report

**TODO:** compute the correlation between orders and wind speed on all 14 days, then on just the 3 surge days. Print both.

Then write one sentence you would actually say in the presentation. The three-day number will look impressive; think hard about whether it means anything.

In [ ]:
# TODO

**What I would say in the presentation:** _..._

### Build 4 — the chart

**TODO:** plot orders by date, mark landfall with a vertical line, and title it with the finding rather than the topic. This chart goes in your deck, so apply the rules from week 12's rubric now rather than rebuilding it later.

In [ ]:
import matplotlib.pyplot as plt
# TODO

---

## Checkpoint (participation)

Report your baseline and surge averages, the multiple between them, and whether your join preserved every sales day.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [ ]:
# Checkpoint
baseline_per_day = None    # TODO
surge_per_day = None       # TODO
multiple = None            # TODO: surge / baseline
rows_preserved = None      # TODO: True if the merge kept all 14 days

print('baseline:', baseline_per_day, 'orders/day')
print('surge:', surge_per_day, 'orders/day')
print('multiple:', multiple)
print('all sales days preserved:', rows_preserved)